# Make range SVG (alpha-shape + convex-hull fallback)

This notebook reproduces the logic from `scripts/make_range_svg.py` and produces a publication-ready SVG of the species' range.

Workflow:
- Install required packages (if needed)
- Load and clean the iNaturalist CSV
- Compute an alpha-shape (concave hull) with convex-hull fallback
- Plot basemap + translucent points + range polygon
- Save and display an SVG

In [ ]:
%pip install -q -r ../requirements.txt

In [39]:
from pathlib import Path
import warnings
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
from IPython.display import SVG, display

# Editable inputs
CSV_PATH = Path('../data/amanita_phalloides/iNat/observations-690863.csv')  # change as needed
OUT_SVG = Path('../outputs/conus_beautified/range.svg')

# Styling / projection (expanded extent to include a bit of Canada and Mexico)
# Expanded by ~5-10 degrees to include Florida/Texas fully and some adjacent Canada/Mexico
LAT_MIN, LAT_MAX = 19.0, 50.0
LON_MIN, LON_MAX = -130.0, -60.0
TARGET_CRS = 'EPSG:5070'
DPI = 240
FIGSIZE = (10.0, 8.0)
LAND_COLOR = '#f3efe3'
WATER_COLOR = '#eaf2f8'
BORDER_COLOR = '#6c7a89'
POINT_OLD_COLOR = '#0080ffff'
POINT_NEW_COLOR = '#c0392b'
S_POINT = 20


In [40]:
def load_and_clean(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found at {csv_path}")
    df_raw = pd.read_csv(csv_path, low_memory=False)
    df = df_raw.copy()
    if {'latitude', 'longitude'}.issubset(df.columns):
        df['lat'] = pd.to_numeric(df['latitude'], errors='coerce')
        df['lon'] = pd.to_numeric(df['longitude'], errors='coerce')
    elif 'location' in df.columns:
        parts = df['location'].astype(str).str.split(',', n=1, expand=True)
        df['lat'] = pd.to_numeric(parts[0], errors='coerce')
        df['lon'] = pd.to_numeric(parts[1], errors='coerce')
    else:
        raise KeyError('Need latitude/longitude columns or a location column ("lat,lon").')

    date_col = None
    for c in ['time_observed_at', 'observed_on', 'observed_on_string', 'created_at', 'time_created_at']:
        if c in df.columns:
            date_col = c
            break
    if date_col is None:
        warnings.warn('Could not find a usable date column; continuing without date-based fields')
        df['date'] = pd.NaT
    else:
        df['date'] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    df = df.dropna(subset=['lat', 'lon']).copy()
    if len(df) == 0:
        raise ValueError('No rows after dropping missing coordinates')
    df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    df = df[df['lat'].between(LAT_MIN, LAT_MAX) & df['lon'].between(LON_MIN, LON_MAX)].copy()
    if len(df) == 0:
        raise ValueError('No rows remain after CONUS bbox filter.')
    return df


In [41]:
def make_geodf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df['lon'], df['lat'])],
        crs='EPSG:4326'
    )
    return gdf


In [42]:
def plot_and_save(gdf_proj: gpd.GeoDataFrame, out_path: Path, taxon_label: str | None):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    ax.set_facecolor(WATER_COLOR)
    fig.patch.set_facecolor('white')

    world = gpd.read_file("https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson")
    bbox_geom = box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
    world_clip = world[world.intersects(bbox_geom)].to_crs(TARGET_CRS)
    world_clip.plot(ax=ax, color=LAND_COLOR, edgecolor=BORDER_COLOR, linewidth=0.7, zorder=1)

    # Points-only mapping: no range polygon is drawn
    ax.scatter(
        gdf_proj.geometry.x,
        gdf_proj.geometry.y,
        s=S_POINT,
        alpha=0.75,
        c=POINT_OLD_COLOR,
        linewidths=0,
        zorder=3,
    )

    if taxon_label:
        ax.text(
            0.5, 1.04,
            taxon_label,
            transform=ax.transAxes,
            ha='center', va='bottom',
            fontsize=16,
            fontstyle='italic',
            color='#2c3e50',
        )

    caption = (
        "Data: iNaturalist export (Research Grade). "
        f"CONUS bbox: lat {LAT_MIN:.2f}–{LAT_MAX:.2f}, lon {LON_MIN:.2f}–{LON_MAX:.2f}. "
        f"Projection: {TARGET_CRS}."
    )
    ax.text(
        0.01, 0.01, caption,
        transform=ax.transAxes,
        fontsize=8.5,
        va='bottom', ha='left',
        color='#2c3e50',
        alpha=0.85,
    )

    # Set axis limits using the geographic bbox projected to the target CRS
    bbox_proj = gpd.GeoSeries([bbox_geom], crs='EPSG:4326').to_crs(TARGET_CRS)
    minx, miny, maxx, maxy = bbox_proj.total_bounds
    pad = 200000  # ~200 km padding
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    ax.set_aspect('equal', adjustable='box')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close(fig)


In [43]:
# Run end-to-end example (edit CSV_PATH / OUT_SVG above if needed)
df = load_and_clean(CSV_PATH)
gdf = make_geodf(df)
gdf_proj = gdf.to_crs(TARGET_CRS)
taxa = sorted(df['scientific_name'].dropna().unique().tolist()) if 'scientific_name' in df.columns else []
taxon_label = taxa[0] if taxa else None
plot_and_save(gdf_proj, OUT_SVG, taxon_label)
print('Wrote:', OUT_SVG.resolve())
display(SVG(str(OUT_SVG)))


Wrote: /Users/jasonraiti/Documents/GitHub/iNat-Mapping/outputs/conus_beautified/range.svg
